# Sarvam Prism vs Jev on Colab
Add SARVAM_API_KEY and OPENROUTER_API_KEY under the key icon (Secrets). No GPU needed: Prism runs through Sarvam's API (its weights need ~120 GB of FP8 GPU memory).

In [ ]:
!pip -q install -U "sarvamai>=0.1.43" requests pandas
import os
from google.colab import userdata
for k in ["SARVAM_API_KEY","OPENROUTER_API_KEY"]:
    try: os.environ[k] = userdata.get(k)
    except Exception as e: print(k, "missing:", e)

In [ ]:
%%writefile decide.py
"""One interface for System One decision models.
backend "sarvam"     -> Sarvam Prism via the Sarvam API  (needs SARVAM_API_KEY)
backend "openrouter" -> Jev or any decision model on OpenRouter's Decisions API
                        (typesafe/jev-1.14, microsoft/microsoft-decision-1,
                        cloudflare/clef, perplexity/..., needs OPENROUTER_API_KEY)
backend "llm"        -> baseline: a normal chat LLM asked to pick a label
Every backend returns {question_id: {"type", "top", "probs"}} so they score
the same way."""
import json, os, re, time, requests

OR_URL = "https://openrouter.ai/api/alpha/decisions"


def _norm(answers):
    out = {}
    for qid, a in answers.items():
        a = a if isinstance(a, dict) else (a.model_dump() if hasattr(a, "model_dump") else vars(a))
        t = a.get("type")
        if t in ("noul", "yes_no"):
            p = a.get("noul", a.get("yes"))
            out[qid] = {"type": "noul", "top": p >= 0.5, "probs": {"yes": p, "no": 1 - p}}
        else:
            probs = a.get("probabilities") or {}
            top = a.get("choice") if t == "choice" else max(probs, key=probs.get)
            out[qid] = {"type": t, "top": top, "probs": probs}
    return out


def decide(backend, model, state, questions):
    t = time.perf_counter()
    if backend == "sarvam":
        from sarvamai import SarvamAI
        c = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
        r = c.decisions.create(model=model, state=state, questions=questions)
        ans = r.answers
    elif backend == "openrouter":
        r = requests.post(OR_URL, timeout=60, json={"model": model, "state": state, "questions": questions},
                          headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
        r.raise_for_status()
        ans = r.json()["answers"]
    elif backend == "llm":
        ans = _llm(model, state, questions)
    else:
        raise ValueError(backend)
    return _norm(ans), (time.perf_counter() - t) * 1000


def _llm(model, state, questions):
    """Baseline the decision models are meant to replace: prompt + parse."""
    spec = {q: {"instructions": v["instructions"],
                "options": list(v["criteria"]) if v["type"] == "choice" else ["yes", "no"]}
            for q, v in questions.items()}
    prompt = (f"State:\n{json.dumps(state)}\n\nAnswer each question with one option. "
              f"Questions: {json.dumps(spec)}\nReturn JSON {{question_id: option}} only.")
    r = requests.post("https://openrouter.ai/api/v1/chat/completions", timeout=60,
                      headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
                      json={"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": 0})
    r.raise_for_status()
    picks = json.loads(re.search(r"\{.*\}", r.json()["choices"][0]["message"]["content"], re.S).group(0))
    out = {}
    for q, v in questions.items():
        p = str(picks.get(q, "")).lower()
        if v["type"] == "noul":
            out[q] = {"type": "noul", "noul": 1.0 if p.startswith("y") else 0.0}
        else:
            out[q] = {"type": "choice", "choice": p, "probabilities": {k: float(k == p) for k in v["criteria"]}}
    return out


In [ ]:
%%writefile run_eval.py
"""Sarvam Prism vs Jev vs other decision models vs an LLM baseline.
Measures: accuracy, calibration (expected vs actual error at >=0.9),
order invariance (shuffle options), p50/p95 latency.
Usage: python run_eval.py   (edit MODELS below; skip ones you have no key for)"""
import json, random, statistics as st
from decide import decide

MODELS = [  # (label, backend, model id)
    ("Sarvam Prism", "sarvam", "sarvam-prism"),
    ("Jev 1.14", "openrouter", "typesafe/jev-1.14"),
    ("Microsoft-Decision-1", "openrouter", "microsoft/microsoft-decision-1"),
    ("Clef", "openrouter", "cloudflare/clef"),
    ("LLM baseline", "llm", "openrouter/auto"),
]
TEAMS = {"billing": "payments, charges, invoices and refunds",
         "tech": "app, website, login or API problems",
         "delivery": "shipping, tracking and damaged parcels"}


def questions(order):
    return {"team": {"type": "choice", "instructions": "Which team should handle this ticket?",
                     "criteria": {k: TEAMS[k] for k in order}},
            "urgent": {"type": "noul", "instructions": "Does this need a reply within the hour?"}}


def evaluate(label, backend, model, rows):
    hits, conf_hits, conf_n, flips, lat = 0, 0, 0, 0, []
    for r in rows:
        a, ms = decide(backend, model, r["state"], questions(list(TEAMS)))
        lat.append(ms)
        ok = a["team"]["top"] == r["team"] and a["urgent"]["top"] == r["urgent"]
        hits += ok
        p = a["team"]["probs"].get(a["team"]["top"], 0)
        if p >= 0.9:
            conf_n += 1; conf_hits += a["team"]["top"] == r["team"]
        shuffled = list(TEAMS); random.shuffle(shuffled)
        b, _ = decide(backend, model, r["state"], questions(shuffled))
        flips += b["team"]["top"] != a["team"]["top"]
    n = len(rows); lat.sort()
    return {"model": label, "accuracy": round(hits / n, 3),
            "answers_at_0.9": f"{conf_n}/{n}",
            "error_at_0.9": round(1 - conf_hits / conf_n, 3) if conf_n else None,
            "flips_when_shuffled": f"{flips}/{n}",
            "p50_ms": round(st.median(lat)), "p95_ms": round(lat[int(0.95 * (n - 1))])}


if __name__ == "__main__":
    rows = [json.loads(l) for l in open("tickets.jsonl")]
    results = []
    for m in MODELS:
        try:
            results.append(evaluate(*m, rows)); print(results[-1])
        except Exception as e:
            print(m[0], "skipped:", e)
    json.dump(results, open("results.json", "w"), indent=2)


In [ ]:
%%writefile tickets.jsonl
{"state": "I was charged twice for my Pro plan this month. Please refund one.", "team": "billing", "urgent": true}
{"state": "The app crashes every time I open the settings page on Android 15.", "team": "tech", "urgent": false}
{"state": "My order #4412 says delivered but nothing arrived at my door.", "team": "delivery", "urgent": true}
{"state": "Can I change the card my subscription is billed to?", "team": "billing", "urgent": false}
{"state": "Checkout has been down for all customers for 30 minutes, we are losing sales.", "team": "tech", "urgent": true}
{"state": "When will my parcel to Pune be shipped? It has been pending for 4 days.", "team": "delivery", "urgent": false}
{"state": "Mera payment do baar cut gaya, please paisa wapas karo.", "team": "billing", "urgent": true}
{"state": "Login OTP is not arriving on my phone, I cannot access my account.", "team": "tech", "urgent": true}
{"state": "The courier left my package with the neighbour, that is fine, just letting you know.", "team": "delivery", "urgent": false}
{"state": "Your invoice PDF shows the wrong GST number for my company.", "team": "billing", "urgent": false}
{"state": "Dark mode text is hard to read on the dashboard.", "team": "tech", "urgent": false}
{"state": "The box arrived crushed and the glass inside is broken.", "team": "delivery", "urgent": true}
{"state": "I cancelled last week but was still charged today. This is the third time I am writing.", "team": "billing", "urgent": true}
{"state": "Export to CSV gives a 500 error, JSON export still works.", "team": "tech", "urgent": false}
{"state": "Tracking has not updated since it left the Mumbai hub on Monday.", "team": "delivery", "urgent": false}
{"state": "माझे पेमेंट अयशस्वी झाले पण खात्यातून पैसे कापले गेले.", "team": "billing", "urgent": true}
{"state": "The API returns 429 errors even though we are under our plan limit, production is blocked.", "team": "tech", "urgent": true}
{"state": "Can you deliver to a PO box address?", "team": "delivery", "urgent": false}
{"state": "Do you offer an annual plan discount?", "team": "billing", "urgent": false}
{"state": "Password reset link says expired immediately after I click it.", "team": "tech", "urgent": false}


In [ ]:
import json, pandas as pd, run_eval
rows = [json.loads(l) for l in open('tickets.jsonl')]
res = []
for m in run_eval.MODELS:
    try: res.append(run_eval.evaluate(*m, rows))
    except Exception as e: print(m[0], 'skipped:', e)
pd.DataFrame(res)